# Learning Objectives

* Build a federated-learning reproduction workflow — FL rounds as Pegasus sub-workflows — from Python wrappers and Apptainer containers.
* Generate the data, training, evaluation and comparison stages through the Pegasus Python API, using the same generator class as the command-line tool.
* Visualize a workflow DAG before submitting it.
* Plan, submit, monitor, and pull statistics for a Pegasus workflow from a notebook.
* Inspect the results: TOPSIS rankings, per-event metrics and the validation report.

# Fed-Cast: Federated Precipitation Nowcasting Workflow

This workflow reproduces Fed-Cast (Xu et al., eScience 2026): federated vs. centralized DGMR precipitation
nowcasting on MRMS PrecipRate over seven radar-centered clients, scored with MCT-style metrics and TOPSIS
against a PySTEPS STEPS baseline. Data are fetched from the public MRMS archive at run time; FL rounds run as
per-round sub-workflows. This notebook runs the **pilot** configuration (`--test`: 2 radar sites, 1 month,
2 rounds) — the same settings as `./workflow_generator.py --test`. See `README.md` and `SPEC.md` for the full
design, placement modes and outputs.

**Workflow Jobs:**
1. **Data** — fetch and crop MRMS per (site, month), build training sequences per site
2. **Centralized baseline** — pool every client's data and train DGMR in segments
3. **Federated arm** — per-round sub-workflows: local training on each client, then aggregation
4. **Evaluation** — best checkpoints, benchmark events, MCT metrics vs. the STEPS baseline
5. **Comparison** — TOPSIS per experiment pool, figures and the validation report

**Prerequisites:** [Pegasus WMS](https://pegasus.isi.edu/) >= 5.0, [HTCondor](https://htcondor.org/) >= 10.2,
Python 3.8+ with `pyyaml`, and [Apptainer](https://apptainer.org/) — all assumed to already be available in
this notebook's environment.

## 1. Build the Apptainer Container

The workflow's jobs run in three Apptainer containers — data (`FedCast_data`), training (`FedCast_train`,
GPU) and evaluation (`FedCast_eval`). Building them takes a while; the cell below skips any `.sif` that
already exists.

In [ ]:
%%bash
for img in FedCast_data FedCast_train FedCast_eval; do
    if [ ! -f Apptainer/$img.sif ]; then
        apptainer build Apptainer/$img.sif Apptainer/$img.def
    else
        echo "Container already built: Apptainer/$img.sif"
    fi
done

## 2. Generate the Workflow

The following cell uses `workflow_generator.py` — the same generator used by the command-line tool — to
build the parent workflow, the FL-round sub-workflows, and their catalogs. The inputs go through the CLI's own
argument parser and pilot/validation step (`build_parser()`, `finalize_args()`), so nothing about the workflow
is defined here: change the pipeline in `workflow_generator.py`, and both the notebook and the CLI pick it up.

`create_sites_catalog()` writes a self-contained site catalog for this machine's HTCondor pool (`local` +
`compute`) to `sites.yml`. GPU jobs carry the `gpu` tag; on a plain pool they run wherever HTCondor matches
them. To run on a resource with a centrally hosted site catalog
([pegasus-site-catalogs](https://github.com/pegasushub/pegasus-site-catalogs/tree/main/conf), e.g.
`unity.yml`), set `HOSTED_SITE_CATALOG` instead — or set it once in `~/.pegasusrc` — and add your
allocation with `./custom_sites.py` (see "Sites" in `README.md`).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))

from workflow_generator import (FedCastWorkflow, build_parser,
                                check_site_catalog_setup, finalize_args)

EXEC_SITE = "compute"
HOSTED_SITE_CATALOG = None  # e.g. "unity.yml"; None = local HTCondor catalog below

# The same arguments the CLI takes (see README.md): the pilot run
cli = ["--test", "--output", "workflow.yml"]
if HOSTED_SITE_CATALOG:
    cli += ["-s", HOSTED_SITE_CATALOG]
args = finalize_args(build_parser().parse_args(cli))

workflow = FedCastWorkflow(args)
workflow.log_placement()

if HOSTED_SITE_CATALOG is None:
    print("Creating execution sites...")
    workflow.create_sites_catalog(exec_site_name=EXEC_SITE)

# The same sequence as the CLI's main()
LEAF_CLEANUP = check_site_catalog_setup(args, workflow.silos)
print("Creating workflow properties...")
workflow.create_pegasus_properties()
workflow.set_container_binds(LEAF_CLEANUP)

print("Creating transformation catalog...")
workflow.create_transformation_catalog()

print("Creating replica catalog...")
workflow.create_replica_catalog()

print("Creating Fed-Cast workflow DAG and FL-round sub-workflows...")
workflow.write_subworkflow_conf()
workflow.create_workflow()

workflow.write()
print("\nFed-Cast Workflow has been generated!")

## 3. View the Generated Workflow DAG

Before submitting, visualize the parent workflow DAG using `pegasus-graphviz`. The graph shows the per-site
data jobs, the centralized training segments, one sub-workflow node per FL round, and the evaluation and
TOPSIS jobs that gather both arms.

In [ ]:
!pegasus-graphviz -f workflow.yml --output workflow.png

In [ ]:
from IPython.display import Image

Image(filename="workflow.png")

## 4. Plan and Submit the Workflow

The workflow will be planned and submitted for execution on the site chosen above (`EXEC_SITE`).

The CLI version of this generator (`workflow_generator.py`) writes the same workflow non-interactively and
prints the `pegasus-plan` command; it never submits by itself. Submitting is this explicit step.

In [ ]:
workflow.plan_submit(exec_site_name=EXEC_SITE,
                     cleanup="leaf" if LEAF_CLEANUP else "none")

## Workflow Status Monitoring

After successful submission, monitor the workflow status. The output shows job counts and idle/running/completed
states. `wait()` blocks (with a progress bar) until the workflow completes or fails.

In [ ]:
workflow.status()

In [ ]:
workflow.wait()

## 5. Statistics

Once the workflow completes, pull execution statistics from the Pegasus provenance database.

In [ ]:
workflow.statistics()

## 6. Examine the Results

The workflow stages the following final outputs to `output/` (`--output-dir`):

| Output | Description |
|--------|-------------|
| `{site}_manifest.json` | Frozen split manifests with SHA-256 |
| `benchmark_sequences.npz` | The benchmark event samples, as evaluated |
| `{method}_L{L}_{YYYYMM}_best.ckpt` | Best-validation-loss checkpoints |
| `{method}_L{L}_{YYYYMM}_metrics.csv`, `steps_metrics.csv` | Per-event metrics |
| `e1_topsis.csv` (+ `e21`/`e22_*`) | TOPSIS scores per experiment pool |
| `figures.tar.gz` | Learning curves and summary table |
| `validation_report.md` | SPEC Sec. 5 gate results |

In [ ]:
!ls -ltR output/

Let's look at the TOPSIS ranking of the trained models (higher is better):

In [ ]:
import pandas as pd

pd.read_csv("output/e1_topsis.csv")